# v0.3 S3-NQ — Llama reader on NQ (E2) and the full top-k ablation with Qwen (E4)
**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**. It runs unattended (about 3.5–4.5 h). If a step fails, the notebook stops with `FAILED` — send me the log. When it finishes, open the version's **Output** tab, download `v03_s3nq_results.zip` and send it to me.

**Input:** Add Input → **Your Work → Notebooks** → the v0.2 **Main S1** notebook from 27 Sep (its output has `main_s1_results.zip`). If it does not appear, upload `main_s1_results.zip` as a private dataset and attach that.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 0b — run two commands at once (one per GPU); STOP if either fails
import threading
def sh_par(*cmds):
    errs = []
    def run(c):
        try:
            sh(c)
        except Exception as e:
            errs.append(str(e))
    ts = [threading.Thread(target=run, args=(c,)) for c in cmds]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errs:
        raise RuntimeError("; ".join(errs))

In [ ]:
# Cell 0c — the code of commit 33394ed as base64 tar.gz (do not edit; Cell 1 checks it)
CODE = "..."

In [ ]:
# Cell 1 — hardware, code frozen at 33394ed (embedded in Cell 0c), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
try:                                   # Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("HF_TOKEN not available (public downloads still work):", type(e).__name__)
# the GitHub repo is private: the code of commit 33394ed346732ae500752d9fd142d93a550e3bf3 is embedded in Cell 0c (no dataset, no clone);
# STOP unless the extracted files reproduce that commit's git tree exactly (paper_inputs, paper_outputs left out)
import base64, io, shutil, tarfile
shutil.rmtree("rag", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE))).extractall("rag")
os.chdir("/kaggle/working/rag")
sh("git init -q && git add -A -f")
sh("test \"$(git write-tree)\" = 4555317fe50de5ad69e352bd167b323a178d6f85 || (echo WRONG CODE; exit 1)")
sh("git -c user.name=kaggle -c user.email=kaggle@local commit -qm 'code 33394ed346732ae500752d9fd142d93a550e3bf3' && echo CODE OK 33394ed")
sh("pip install -q -r requirements-v03-retrieval.txt")
sh("pip install -q -r requirements-v03-generation.txt")
sh("pip install -q --force-reinstall --no-deps torchaudio==2.11.0")
sh("python -c \"import torch, torchaudio, torchvision, vllm, transformers; print(torch.__version__, torch.version.cuda, torchaudio.__version__, torchvision.__version__, vllm.__version__, transformers.__version__); assert torch.cuda.is_available() and torchaudio.__version__.endswith(torch.version.cuda.replace(\x27.\x27, \x27\x27)), \x27torch/torchaudio CUDA mismatch\x27\"")

sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — restore the v0.2 S1 output FIRST (NQ sub-corpus, joined queries, the 7 retriever runs), then environment
# record, then the NQ corpus text at the v0.2 pinned revision (passage text for the reader)
sh("python -m src.restore --name main_s1_results")
sh("python -m src.check_env --lock requirements-generation.lock --report-only results/raw/check_env_v03_S3nq.json")
sh("python -m src.env_report --session v03_S3nq --out results/raw/env_v03_S3nq.json")
sh("python -m src.data.download --config configs/v03_nq_qwen.yaml --only nq_corpus")

In [ ]:
# Cell 3 — in parallel: GPU 0 = E4 top-k with Qwen (R2, R3, R5, R6 x k = 1, 3, 10 x 2,255 queries);
#                       GPU 1 = E2 Llama-3.1-8B on NQ (9 conditions x 2,255 queries, k = 5)   (~3-3.5 h)
GEN_ERR = ""
try:
    sh_par("CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/v03_nq_qwen.yaml --conditions R2,R3,R5,R6 --corpus main --n 2255 --k 1 --auto-eager-retry && CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/v03_nq_qwen.yaml --conditions R2,R3,R5,R6 --corpus main --n 2255 --k 3 --auto-eager-retry && CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/v03_nq_qwen.yaml --conditions R2,R3,R5,R6 --corpus main --n 2255 --k 10 --auto-eager-retry",
           "CUDA_VISIBLE_DEVICES=1 python -m src.generation.run --config configs/v03_nq_llama.yaml --conditions R0,R1,R2,R3,R4,R5,R6,CB,GOLD --corpus main --n 2255 --k 5 --auto-eager-retry")
except Exception as e:
    GEN_ERR = str(e); print("READER FAILURE (recorded; finished conditions are still packaged):", GEN_ERR)

In [ ]:
# Cell 4 — score answers (EM, F1, answer containment)
sh("python -m src.evaluation.answer --config configs/v03_nq_qwen.yaml --prefix v03_nq_qwen")
sh("python -m src.evaluation.answer --config configs/v03_nq_llama.yaml --prefix v03_nq_llama")

In [ ]:
# Last cell — package: everything EXCEPT the large run files (they stay in the earlier output) -> v03_s3nq_results.zip
# small (about 20-40 MB): send this file to Claude
import zipfile
from pathlib import Path
n = 0
with zipfile.ZipFile("/kaggle/working/v03_s3nq_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in ['results/raw', 'results/processed', 'configs/resolved_revisions.json', 'logs']:
        for p in ([Path(k)] if Path(k).is_file() else sorted(Path(k).rglob("*"))):
            if p.is_file() and "/runs/" not in p.as_posix() and (not k.startswith("data/") or p.name.startswith(())):
                z.write(p); n += 1
print(n, "files"); sh("ls -la /kaggle/working/v03_s3nq_results.zip")
if GEN_ERR:
    raise RuntimeError("FAILED (packaged anyway): " + GEN_ERR)